# 📡 **TellCo User Analytics**

### *Importing Final Scores into SQL Server*

---

**Author**  : Yousuf S. R. Sakkaf

**Status**  : Completed

**Dataset** : user_satisfaction_scores.csv

**File**    : telecom_SQL_Import.ipynb

**Project** : NHIS_Project5

**Date**    : 5th August 2026

---

## SQL Import

<div style="margin-left: 20px;max-width: 90%">

Reads the final scored table (`user_satisfaction_scores.csv`, exported from the main analysis notebook) and imports it into a local SQL Server Express database via Windows Authentication, using `sqlalchemy` and `pyodbc`. A `SELECT` query against the imported table confirms success.

</div>

> **Note:** This notebook must be run locally (not Google Colab). Windows Authentication requires execution on the same machine as the SQL Server Express instance — Colab's cloud runtime has no network path to a local machine's SQL Server.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from IPython.display import display, HTML

PRIMARY_COLOR   = "#005B96"
SECONDARY_COLOR = "#EAF4FC"
SUCCESS_COLOR   = "#2E8B57"
WARNING_COLOR   = "#C0392B"
INFO_COLOR      = "#1F77B4"
TABLE_COLOR     = "#5B88B2"
PLOT_COLOR      = "#262661"
PILLAR_COLOR    = "#0E7C7B"
# Display Functions


def section_header(title, pillar=False):
    bg = PILLAR_COLOR if pillar else PRIMARY_COLOR
    display(HTML(f"""
    <div style="background:{bg};color:white;padding:16px;border-radius:8px;font-size:28px;font-weight:bold;margin-top:20px;margin-bottom:12px;text-align:center;">
        {title}
    </div>"""))

def subsection_header(title):
    display(HTML(f"""
    <div style="background:{SECONDARY_COLOR};border-left:6px solid {PRIMARY_COLOR};color:{PRIMARY_COLOR};padding:10px;border-radius:6px;font-size:20px;font-weight:bold;margin-top:15px;margin-bottom:10px;text-align:center;">
        <i>{title}</i>
    </div>"""))

def success_box(message):
    display(HTML(f"""
    <div style="background:#EAF7EA;border-left:6px solid {SUCCESS_COLOR};color:{SUCCESS_COLOR};padding:12px;border-radius:6px;margin:10px 0;">
        <b>✅ Success:</b><br><span style="color:black;">{message}</span>
    </div>"""))

def warning_box(message):
    display(HTML(f"""
    <div style="background:#FDEDEC;border-left:6px solid {WARNING_COLOR};color:{WARNING_COLOR};padding:12px;border-radius:6px;margin:10px 0;">
        <b>⚠️ Note:</b><br><span style="color:black;">{message}</span>
    </div>"""))

def info_box(message):
    display(HTML(f"""
    <div style="background:#F1F3F6;border-left:6px solid {INFO_COLOR};color:{INFO_COLOR};padding:12px;border-radius:6px;margin:10px 0;">
        <b>ℹ️ Information:</b><br><span style="color:black;">{message}</span>
    </div>"""))

In [2]:
def centered_table(df, index=False, float_format='{:.2f}'):
    """Renders a DataFrame as a centered, well-spaced HTML table matching the theme."""
    html = df.to_html(index=index, border=0, escape=False, float_format=float_format.format)
    html = html.replace('class="dataframe"', '')
    html = html.replace(
        '<table', '<table style="margin:auto;border-collapse:separate;border-spacing:0;'
        'border:1px solid #ccc;border-radius:6px;overflow:hidden;"'
    ).replace(
        '<th>', f'<th style="background:{TABLE_COLOR};color:white;padding:10px 24px;'
                f'text-align:center !important;border-bottom:1px solid #ccc;">'
    ).replace(
        '<td>', '<td style="padding:8px 24px;text-align:center !important;border-bottom:1px solid #eee;">'
    )
    display(HTML(f'<div style="overflow-x:auto;margin:15px 0;">{html}</div>'))
success_box("HTML Theme initiated and ready.")

In [3]:
section_header("Loading Final Scores CSV")

user_satisfaction_scores = pd.read_csv('Assets/user_satisfaction_scores.csv')
centered_table(user_satisfaction_scores.head(10), float_format='{:,.3f}')
info_box(f"Loaded {user_satisfaction_scores.shape[0]:,} rows from 'Assets/user_satisfaction_scores.csv'.")

MSISDN/Number,Engagement Score,Experience Score,Satisfaction Score
33601001722,0.471,0.258,0.365
33601001754,0.506,0.236,0.371
33601002511,0.419,0.052,0.235
33601007832,0.180,0.640,0.410
33601008617,0.683,0.495,0.589
33601010682,0.603,0.126,0.364
33601011634,0.196,0.607,0.401
33601011959,0.123,0.196,0.159
33601014694,0.556,0.051,0.303
33601020306,0.283,0.228,0.255


In [4]:
section_header("Data Import to SQL Server")
subsection_header("Connecting to Server & Ensuring Database Exists")

server = 'localhost\\SQLEXPRESS'
database = 'TelecomDB'

# CREATE DATABASE can't run inside a transaction, and requires connecting to an existing database first —
#so 'master' is used only to check/create 'TelecomDB', never to hold project data itself.
master_url = f"mssql+pyodbc://@{server}/master?driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes&Encrypt=yes&TrustServerCertificate=yes"
master_engine = create_engine(master_url, isolation_level="AUTOCOMMIT")

with master_engine.connect() as conn:
    exists = conn.execute(text(f"SELECT database_id FROM sys.databases WHERE name = '{database}'")).fetchone()
    if not exists:
        conn.execute(text(f"CREATE DATABASE {database}"))
        success_box(f"Database '{database}' did not exist and has been created.")
    else:
        info_box(f"Database '{database}' already exists — no action needed.")

master_engine.dispose()

connection_url = f"mssql+pyodbc://@{server}/{database}?driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes&Encrypt=yes&TrustServerCertificate=yes"
engine = create_engine(connection_url)
info_box(f"Connected to '{database}' on server '{server}'.")

In [5]:
subsection_header("Importing into SQL Server")

table_name = 'user_satisfaction_scores'
import_table = user_satisfaction_scores.copy()

import_table.columns = import_table.columns.str.replace('/', '_').str.replace(' ', '_')

import_table.to_sql(table_name, engine, if_exists='replace', index=False)
success_box(f"Imported {import_table.shape[0]:,} rows into SQL Server table '{table_name}' in database '{database}', with SQL-safe column names: {list(import_table.columns)}.")

In [6]:
subsection_header("Verifying the Import")

with engine.connect() as conn:
    result = conn.execute(text(f"SELECT TOP 10 * FROM {table_name}"))
    verify_df = pd.DataFrame(result.fetchall(), columns=result.keys())

centered_table(verify_df, float_format='{:,.3f}')
success_box(f"Database import verified successfully. Live SQL query executed and returned the first 10 records from '{table_name}'.")
engine.dispose()

MSISDN_Number,Engagement_Score,Experience_Score,Satisfaction_Score
33601001722,0.471,0.258,0.365
33601001754,0.506,0.236,0.371
33601002511,0.419,0.052,0.235
33601007832,0.180,0.640,0.410
33601008617,0.683,0.495,0.589
33601010682,0.603,0.126,0.364
33601011634,0.196,0.607,0.401
33601011959,0.123,0.196,0.159
33601014694,0.556,0.051,0.303
33601020306,0.283,0.228,0.255


In [7]:
display(HTML(f"""
<div style="text-align:center;margin-top:40px;margin-bottom:30px;">
    <h3 style="color:#2E8B57;"><b>✅ Checkpoint: SQL Import Complete</b></h3>

    <p style="font-style:italic;font-size:1.05em;opacity:0.85;">
        The final table (customer ID with Engagement, Experience, and Satisfaction scores) has been
        imported into 'TelecomDB' on the local SQL Server Express instance, with SQL-safe column names
        and a verified SELECT query confirming successful import. A separate SSMS screenshot of this
        query and its output is used for the accompanying PPT.
    </p>

    <hr style="width:45%;">
</div>"""))